# Benchmark — 3 kategori × 3 zorluk, tek çalıştırma, W&B'ye loglanır

Yayınlanan veri kümesiyle ([`eylulpelinkilic/Colored_Point_Clouds`](https://huggingface.co/datasets/eylulpelinkilic/Colored_Point_Clouds))
bütün ızgarayı **tek Run all** ile dolaşır.

**İki eksende kıyas:**

| eksen | ne ölçülüyor |
|---|---|
| **PoinTr'ın kendi başarımı** | `chamfer/partial→gt` vs `chamfer/completion→gt` — tamamlama ham girdiden ne kadar iyi, zorlukla nasıl bozuluyor |
| **Renklendirme yöntemleri** | NN-kopya (PoinTr + naif renk) baseline; part-mean, RePaint-vanilla, RePaint-part, RePaint-part (oracle seg) ona karşı |

**Neden ızgara ucuz.** RePaint'in DDPM'i **occlusion maskesi görmeden** eğitiliyor —
kategori başına bir kez eğitilip üç zorlukta da kullanılıyor. 9 eğitim değil 3.
Bu zaten yöntemin satış argümanı; tablo onu doğrudan gösteriyor.

**Kopmaya dayanıklı.** Eğitilmiş modeller Drive'a, her test modelinin sonucu anında
`results.jsonl`'a yazılır. Runtime düşerse **aynı hücreyi tekrar çalıştır** — eğitim ve
hesaplanmış modeller atlanır, kaldığı yerden sürer. HF indirmeleri de Drive'da önbelleklenir.

```
kategori başına: veri + frame + part-seg + 2 DDPM ≈ 30 dk
                 değerlendirme 3 zorluk × 50 model  ≈ 110 dk
üç kategori toplam ≈ 6-7 saat
```

Occlusion **yeniden üretilmiyor** — HF'de yayınlanan `occluded_occ` partial'ları
kullanılıyor, yani sonuçlar veri kümesinden birebir tekrar üretilebilir.

---
# A · Ortam kurulumu
> `RePaint_part_colab.ipynb` ile **birebir aynı**.

In [ ]:
!nvidia-smi -L
# GPU görünmüyorsa: Runtime > Change runtime type > Hardware accelerator = GPU

### 1) Python bağımlılıkları (numpy<2 sabit; open3d/timm güncel)

In [ ]:
# torch 2.11 numpy 2.x'e karşı derli -> numpy'yi DOWNGRADE ETME (mixed-install -> mtrand ABI hatası).
# Pinli eski open3d==0.9 / timm==0.4.5 py3.12'de derlenmez; güncelleri kurulur.
!pip install -q easydict h5py matplotlib opencv-python pyyaml scipy \
    tensorboardX tqdm transforms3d einops timm open3d gdown
import numpy as np; print("deps OK | numpy", np.__version__, "(2.x olmalı)")

> ⚠️ **numpy'yi 2.x'te bırak** (torch 2.11 onu ister). Eğer `numpy.dtype size changed` /
> `mtrand` hatası alırsan numpy karışmış demektir → şunu çalıştır ve **Runtime ▸ Restart**:
> `!pip install --force-reinstall --no-cache-dir "numpy==2.0.2"`  — sonra baştan çalıştır.

### 2) Fork'u klonla → `/content/PoinTr`

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/PoinTr"):
    !git clone https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git /content/PoinTr
%cd /content/PoinTr
!git rev-parse --short HEAD

### 3) CUDA build ortamı (sabit değil — otomatik tespit + doğru GPU arch)

In [ ]:
import os, glob, torch
# Colab'da aktif toolkit /usr/local/cuda sembolik linkidir; sürümü hardcode ETME
cuda_home = "/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else sorted(glob.glob("/usr/local/cuda*"))[-1]
os.environ["CUDA_HOME"] = cuda_home
os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
# eklentiler DOĞRU GPU mimarisi için derlensin (T4=7.5, V100=7.0, A100=8.0, L4=8.9) -> "no kernel image" hatasını önler
cap = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
print("torch", torch.__version__, "| torch-cuda", torch.version.cuda,
      "| CUDA_HOME", cuda_home, "| arch", os.environ["TORCH_CUDA_ARCH_LIST"])
!nvcc --version | tail -2

### 4) `pointnet2_ops` (saf-PyTorch shim — derleme yok)
PoinTr'ın `fps`/`three_nn` gibi ops'ları buna bağlı. Çok yeni torch'ta eski CUDA
repo'su derlenmediği için, kullanılan 6 fonksiyonu saf torch'la enjekte ediyoruz.

In [ ]:
# pointnet2_ops'u DERLEMEK yerine saf-PyTorch SHIM olarak enjekte ediyoruz.
# torch 2.11+cu128 gibi çok yeni stack'te eski CUDA repo'su derlenmiyor. PoinTr sadece
# şu 6 fonksiyonu kullanıyor; hepsi saf torch'la doğru (yerelde brute-force'a karşı test edildi).
# FPS saf-torch döngüsü biraz yavaş ama bu ölçekte (~8k nokta) sorun değil.
import sys, types, torch

def furthest_point_sample(xyz, npoint):        # xyz (B,N,3) -> idx (B,npoint) int32
    B, N, _ = xyz.shape; dev = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev, dtype=xyz.dtype)
    far = torch.zeros(B, dtype=torch.long, device=dev); ar = torch.arange(B, device=dev)
    for i in range(npoint):
        idx[:, i] = far
        dist = torch.minimum(dist, ((xyz - xyz[ar, far].unsqueeze(1)) ** 2).sum(-1))
        far = torch.max(dist, dim=1).indices
    return idx.int()

def gather_operation(features, idx):           # (B,C,N),(B,S) -> (B,C,S)
    B, C, N = features.shape; idx = idx.long()
    return torch.gather(features, 2, idx.unsqueeze(1).expand(B, C, idx.shape[1])).contiguous()

def three_nn(query, ref):                      # (B,N,3),(B,M,3) -> dist(B,N,3) öklid, idx(B,N,3)
    d = torch.cdist(query, ref)
    dist, idx = torch.topk(d, 3, dim=-1, largest=False, sorted=True)
    return dist.contiguous(), idx.int().contiguous()

def three_interpolate(features, idx, weight):  # (B,C,M),(B,N,3),(B,N,3) -> (B,C,N)
    B, C, M = features.shape; N = idx.shape[1]; idx = idx.long()
    g = torch.gather(features, 2, idx.reshape(B,1,N*3).expand(B,C,N*3)).reshape(B,C,N,3)
    return (g * weight.unsqueeze(1)).sum(-1).contiguous()

def grouping_operation(features, idx):         # (B,C,N),(B,S,K) -> (B,C,S,K)  (SnowFlakeNet için)
    B, C, N = features.shape; _, S, K = idx.shape; idx = idx.long()
    return torch.gather(features, 2, idx.reshape(B,1,S*K).expand(B,C,S*K)).reshape(B,C,S,K).contiguous()

def ball_query(radius, nsample, xyz, new_xyz): # (r,k,(B,N,3),(B,S,3)) -> idx(B,S,k)  (SnowFlakeNet için)
    B, N, _ = xyz.shape; S = new_xyz.shape[1]
    d = torch.cdist(new_xyz, xyz)
    idx = torch.arange(N, device=xyz.device).view(1,1,N).expand(B,S,N).contiguous()
    idx[d > radius] = N
    idx = idx.sort(dim=-1).values[:, :, :nsample]
    first = idx[:, :, 0:1].clone(); first[first == N] = 0
    idx = torch.where(idx == N, first.expand(-1,-1,nsample), idx)
    return idx.int()

_u = types.ModuleType("pointnet2_ops.pointnet2_utils")
for _f in [furthest_point_sample, gather_operation, three_nn, three_interpolate,
           grouping_operation, ball_query]:
    setattr(_u, _f.__name__, _f)
_p = types.ModuleType("pointnet2_ops"); _p.pointnet2_utils = _u
sys.modules["pointnet2_ops"] = _p
sys.modules["pointnet2_ops.pointnet2_utils"] = _u
from pointnet2_ops import pointnet2_utils
print("pointnet2_ops shim enjekte edildi:",
      [n for n in dir(pointnet2_utils) if not n.startswith("_")])

### 5) CUDA extension'ları derle (`chamfer` zorunlu; gridding/cubic GRNet için)

In [ ]:
import subprocess
EXTS = ["chamfer_dist", "gridding", "gridding_loss", "cubic_feature_sampling"]  # emd PoinTr için gerekmez
for ext in EXTS:
    print(f"── building {ext} ──")
    # --no-build-isolation: bu setup.py'ler de torch.utils.cpp_extension'a bağlı
    r = subprocess.run("pip install -q --no-build-isolation .", shell=True,
                       cwd=f"/content/PoinTr/extensions/{ext}", capture_output=True, text=True)
    ok = r.returncode == 0
    print("   ", "✅ ok" if ok else "‼ FAILED")
    if not ok:
        print(r.stdout[-600:]); print(r.stderr[-1800:])

### 6) Her şey import oluyor mu? (GPU smoke test)

In [ ]:
import os, sys
sys.path.insert(0, "/content/PoinTr"); os.chdir("/content/PoinTr")
import torch, numpy as np
print("numpy", np.__version__, "| torch", torch.__version__)
import chamfer, gridding, gridding_distance, cubic_feature_sampling
from pointnet2_ops import pointnet2_utils
from extensions.chamfer_dist import ChamferDistanceL1
from models.PoinTr import PoinTr, Fold, fps
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
# gerçekten GPU'da çalışıyor mu: fps + chamfer
x = torch.rand(1, 1024, 3, device="cuda")
idx = pointnet2_utils.furthest_point_sample(x, 128)
d = ChamferDistanceL1()(x, torch.rand(1, 512, 3, device="cuda"))
print("pointnet2 fps:", tuple(idx.shape), "| chamfer:", float(d))
print("✅ PoinTr environment READY")

### 7) Pretrained checkpoint (ShapeNet55)

In [ ]:
import os, subprocess
CKPT = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)
print("checkpoint MB:", round(os.path.getsize(CKPT)/1e6, 1), " (>400 olmalı)")

---
# B · Weights & Biases

Hesabın yoksa [wandb.ai](https://wandb.ai) üzerinden ücretsiz aç. `wandb.login()`
API anahtarını soracak — [wandb.ai/authorize](https://wandb.ai/authorize) adresinden alırsın.

In [ ]:
!pip install -q wandb
import wandb, os

# Anahtarı Colab Secrets'tan al -> her restart'ta tekrar sormaz.
#   Sol kenardaki 🔑 -> Add new secret -> Name: WANDB_API_KEY -> Notebook access AÇIK
# Secret yoksa interaktif isteme düşer (anahtarı oraya yapıştır).
_key = None
try:
    from google.colab import userdata
    _key = userdata.get("WANDB_API_KEY")
except Exception:
    _key = os.environ.get("WANDB_API_KEY")

if _key:
    wandb.login(key=_key)
    print("wandb: Secrets'tan giriş yapıldı")
else:
    print("WANDB_API_KEY secret'ı yok -> anahtarı aşağıya yapıştır")
    print("  (hesabın varsa seçenek 2'yi seç; anahtar: https://wandb.ai/authorize)")
    wandb.login()

WANDB_PROJECT = "colored-pc-completion"
WANDB_ENTITY  = None                # None = kendi hesabın; takım varsa adını yaz
print("wandb", wandb.__version__)

---
# C · Veri — yayınlanan HF veri kümesinden

Elle dosya yüklemek yok. `gt_s3` (renkli GT + parça etiketi `labeled_s3`'ten) ve
`occluded_occ` (yayınlanan partial'lar) doğrudan indirilir.

`partial ∪ missing = gt` olduğu için eksik-bölge maskesi partial'ın GT içindeki
karşılıklarından **birebir** çıkarılır; yeniden occlusion üretmiyoruz.

In [ ]:
# ================= BENCHMARK AYARLARI =================
# ÜÇ KATEGORİYİ DE tek çalıştırmada dolaşır. Kopma olursa aynı hücreyi tekrar
# çalıştır: eğitim ve hesaplanmış modeller atlanır, kaldığı yerden sürer.
# Sadece birini istersen listeyi kısalt: RUN_CATEGORIES = ["airplane"]
RUN_CATEGORIES = ["airplane", "car", "chair"]

HF_DATA      = "eylulpelinkilic/Colored_Point_Clouds"
CATEGORIES   = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
DIFFICULTIES = ["simple", "moderate", "hard"]      # %25 / %50 / %75
N_MODELS     = None       # None = TAMAMI (kategori başına ~200)
N_PTS        = 2048       # DDPM/eval çözünürlüğü (kaynak 8192)
TEST_FRAC    = 0.25
EVAL_N       = None       # None = BÜTÜN test modelleri
N_SEEDS      = 1
FORCE_RETRAIN = False     # True -> checkpoint'leri yok say
# ======================================================
assert all(c in CATEGORIES for c in RUN_CATEGORIES), "bilinmeyen kategori"

import numpy as np, os, glob, torch
from huggingface_hub import snapshot_download

def srgb_to_lab(rgb):
    rgb = np.clip(rgb, 0, 1); lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124,0.3576,0.1805],[0.2126,0.7152,0.0722],[0.0193,0.1192,0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883]); d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116*f[:,1]-16, 500*(f[:,0]-f[:,1]), 200*(f[:,1]-f[:,2])], 1)
def deltaE(a, b): return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)

PART_NAMES = {"airplane": ["body","wing","tail","engine"],
              "car":      ["roof","hood","wheel","body"],
              "chair":    ["back","seat","leg","arm"]}

def fetch(category):
    """Bu kategori için labeled_s3 + occluded_occ indir, yerel kökü döndür."""
    syn = CATEGORIES[category]
    root = snapshot_download(HF_DATA, repo_type="dataset", allow_patterns=[
        f"labeled_s3/{syn}/*", f"occluded_occ/*/{syn}/*"])
    return root, syn

def load_split(category, difficulty):
    """-> DATA listesi. GT + parça npz'den, partial YAYINLANAN ply'den."""
    import open3d as o3d
    root, syn = fetch(category)
    files = sorted(glob.glob(os.path.join(root, "labeled_s3", syn, "*.npz")))
    if N_MODELS: files = files[:N_MODELS]
    DATA = []
    for j, f in enumerate(files):
        mid = os.path.splitext(os.path.basename(f))[0]
        pp = os.path.join(root, "occluded_occ", difficulty, syn, mid + ".ply")
        if not os.path.exists(pp): continue
        z = np.load(f)
        gxyz, grgb, gpart = z["xyz"].astype(np.float32), z["rgb"].astype(np.float32), z["part"].astype(np.int64)
        pxyz = np.asarray(o3d.io.read_point_cloud(pp).points, np.float32)
        # partial GT'nin ALT KUMESI -> eksik maskesi birebir cikar
        from scipy.spatial import cKDTree
        d, idx = cKDTree(gxyz).query(pxyz, k=1)
        vis = np.zeros(len(gxyz), bool); vis[idx[d < 1e-6]] = True
        miss = ~vis
        # calisma cozunurlugune indir (gorunur/eksik oranini koruyarak)
        r = np.random.default_rng(j)
        sel = r.choice(len(gxyz), N_PTS, replace=len(gxyz) < N_PTS)
        gxyz, grgb, gpart, miss = gxyz[sel], grgb[sel], gpart[sel], miss[sel]
        c = gxyz.mean(0); gxyz = (gxyz - c) / (np.linalg.norm(gxyz - c, axis=1).max() + 1e-9)
        gt = np.concatenate([gxyz, grgb], 1).astype(np.float32)
        if miss.sum() < 32 or (~miss).sum() < 32: continue
        DATA.append(dict(gt=gt, partial=gt[~miss], miss=miss, gt_part=gpart,
                         partial_dense=None, model_id=mid))
    return DATA
print("veri yukleyici hazir")

---
# D · Yöntem — `RePaint_part_colab.ipynb`'den birebir

Donmuş PoinTr, oryantasyon araması, PointNet part-seg, DDPM şeması, parça-koşullu
denoiser, koşulsuz eğitim ve RePaint çıkarımı aynen alınıyor.

In [ ]:
# --- STEP 1: dondurulmuş orijinal PoinTr ---
import torch
from easydict import EasyDict
from scipy.spatial import cKDTree
from models.PoinTr import PoinTr, fps

DEV = "cuda"
cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
geo = PoinTr(cfg)
sd = torch.load(CKPT, map_location="cpu")
base = sd.get("base_model", sd.get("model", sd))
base = {k.replace("module.", ""): v for k, v in base.items()}
mi, ui = geo.load_state_dict(base, strict=False)
assert len(mi) == 0, f"orijinal PoinTr bekleniyordu, missing={mi[:4]} (0 olmalı)"
for p in geo.parameters(): p.requires_grad_(False)
geo.eval().to(DEV)

# ShapeNet-Part -> PoinTr(ShapeNet-55) frame hizası. Aşağıdaki hücre 48 işaretli
# permütasyonu Chamfer ile tarayıp bunları OTOMATİK ayarlıyor — elle dokunma.
AXIS_PERM, AXIS_SIGN = (2, 1, 0), (1, 1, 1)

@torch.no_grad()
def complete_geometry(partial, perm=None, sign=None):
    """RENKSİZ partial (xyz) -> PoinTr tamamlaması, doğru frame'e çevirip geri çevirerek."""
    perm = AXIS_PERM if perm is None else tuple(perm)
    sign = AXIS_SIGN if sign is None else tuple(sign)
    s = np.asarray(sign, np.float32)
    x = np.ascontiguousarray(partial[:, :3][:, list(perm)] * s)      # x'[:,i] = x[:,perm[i]]*s[i]
    p = torch.from_numpy(x).float().unsqueeze(0).to(DEV)
    fine = geo(p)[1][0, :geo.num_pred].cpu().numpy()
    inv = np.argsort(perm)                                          # ters çevir: x[:,j] = x'[:,inv[j]]*s[inv[j]]
    return np.ascontiguousarray(fine[:, inv] * s[inv])

def chamfer_l1(a, b):
    """Simetrik ortalama en-yakın-komşu mesafesi (düşük = iyi)."""
    d1, _ = cKDTree(b).query(a, k=1)
    d2, _ = cKDTree(a).query(b, k=1)
    return float(d1.mean() + d2.mean())

# PoinTr ShapeNet-55, 8192 noktalı bulutlardan kırpılmış 2048-6144 noktalı partial'larla
# eğitildi. Bizim seyrek partial'ımız (N_PTS=2048, CROP=0.5 -> 1024 nokta) o dağılımın
# ALTINDA kalıyor ve DGCNN grouper'ın kNN komşulukları ~2x geniş düşüyor. Bu yüzden
# PoinTr'a verilen girdiyi renk hattından AYIRIYORUZ: aşağıdaki hücre en iyisini ölçüyor.
POINTR_IN = 2048          # PoinTr'a verilecek nokta sayısı (yoğun partial varsa FPS ile)
USE_DENSE_FOR_POINTR = True

def _fps_np(xyz, n):
    """PoinTr'ın kendi fps'i (pointnet2_ops shim'i üzerinden)."""
    if len(xyz) <= n:
        return np.ascontiguousarray(xyz).astype(np.float32)
    t = torch.from_numpy(np.ascontiguousarray(xyz[:, :3])).float().unsqueeze(0).to(DEV)
    return fps(t, n)[0].cpu().numpy().astype(np.float32)

def pointr_input(d):
    """Bu model için PoinTr'a verilecek xyz — mümkünse yoğun partial, POINTR_IN'e indirilmiş."""
    src = d.get("partial_dense") if USE_DENSE_FOR_POINTR else None
    src = d["partial"][:, :3] if src is None else src
    return _fps_np(src, POINTR_IN) if POINTR_IN and len(src) > POINTR_IN else src

def complete_of(d, perm=None, sign=None):
    """Bu modelin PoinTr tamamlaması. Boru hattının HER yeri bunu kullanmalı."""
    return complete_geometry(pointr_input(d), perm, sign)

def nn_color(partial, comp):                   # BASELINE 1
    _, i = cKDTree(partial[:, :3]).query(comp[:, :3], k=1)
    return partial[i, 3:6]
print(f"dondurulmuş PoinTr hazır | num_pred={geo.num_pred}")

In [ ]:
# --- oryantasyon araması, FONKSİYON olarak (döngüde kategori başına çağrılır) ---
# Doğru ShapeNet-Part -> ShapeNet-55 frame'i 6 permütasyon x 8 işaret = 48 aday
# arasından, GT'ye Chamfer ile ÖLÇÜLEREK seçilir. Göz kararı değil.
import itertools

def find_frame(DATA, IDX, n_probe=3):
    """-> (perm, sign, en_iyi_chamfer, ham_partial_referansi)"""
    probe = [DATA[i] for i in IDX[:n_probe]]
    ref = float(np.mean([chamfer_l1(d["partial"][:, :3], d["gt"][:, :3]) for d in probe]))
    sc = []
    for perm in itertools.permutations(range(3)):
        for sign in itertools.product((1, -1), repeat=3):
            c = [chamfer_l1(complete_geometry(pointr_input(d), perm, sign), d["gt"][:, :3])
                 for d in probe]
            sc.append((float(np.mean(c)), perm, sign))
    sc.sort()
    return sc[0][1], sc[0][2], sc[0][0], ref

print("find_frame hazır")

In [ ]:
# --- STEP 2: PointNet part-seg (PoinTr_setup_colab.ipynb ile aynı model) ---
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class _TNet(nn.Module):
    def __init__(s, k):
        super().__init__(); s.k = k
        s.mlp = nn.Sequential(nn.Conv1d(k,64,1),nn.BatchNorm1d(64),nn.ReLU(),
            nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU(),
            nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
        s.fc = nn.Sequential(nn.Linear(1024,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU(),nn.Linear(256,k*k))
    def forward(s, x):
        B=x.size(0); f=s.mlp(x).max(-1)[0]
        return s.fc(f).view(B,s.k,s.k) + torch.eye(s.k,device=x.device).unsqueeze(0)

class PointNetPartSeg(nn.Module):
    def __init__(s, P):
        super().__init__(); s.itn=_TNet(3)
        s.mlp1=nn.Sequential(nn.Conv1d(3,64,1),nn.BatchNorm1d(64),nn.ReLU(),nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU())
        s.fstn=_TNet(128)
        s.mlp2=nn.Sequential(nn.Conv1d(128,128,1),nn.BatchNorm1d(128),nn.ReLU(),nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
        s.seg=nn.Sequential(nn.Conv1d(1152,512,1),nn.BatchNorm1d(512),nn.ReLU(),
            nn.Conv1d(512,256,1),nn.BatchNorm1d(256),nn.ReLU(),nn.Conv1d(256,P,1))
    def forward(s, x):
        x=x.transpose(1,2); x=torch.bmm(s.itn(x),x)
        f=s.mlp1(x); f=torch.bmm(s.fstn(f),f); pf=f
        g=s.mlp2(f).max(-1,keepdim=True)[0].expand(-1,-1,f.size(-1))
        return s.seg(torch.cat([pf,g],1)).transpose(1,2)

def train_partseg(model, loader, epochs=30, lr=1e-3, device=DEV):
    model.to(device).train(); opt=torch.optim.Adam(model.parameters(),lr); lf=nn.CrossEntropyLoss()
    for ep in range(epochs):
        cor=seen=0; tot=0.0
        for xyz,lab in loader:
            xyz,lab=xyz.to(device),lab.to(device); opt.zero_grad()
            lo=model(xyz); loss=lf(lo.reshape(-1,lo.size(-1)),lab.reshape(-1))
            loss.backward(); opt.step()
            tot+=loss.item()*xyz.size(0); cor+=(lo.argmax(-1)==lab).sum().item(); seen+=lab.numel()
        if ep%10==0 or ep==epochs-1: print(f"  ep{ep:3d} loss {tot/len(loader.dataset):.4f} acc {cor/seen*100:.1f}%")
    return model

@torch.no_grad()
def segment(model, xyz, device=DEV):
    model.eval(); x=np.asarray(xyz,np.float32)[:,:3]
    c=x.mean(0); x=(x-c)/(np.linalg.norm(x-c,axis=1).max()+1e-9)
    return model(torch.from_numpy(x).float().unsqueeze(0).to(device))[0].argmax(-1).cpu().numpy()

class _GTPartDS(Dataset):
    def __init__(s, D): s.D = D
    def __len__(s): return len(s.D)
    def __getitem__(s, i):
        d = s.D[i]
        return torch.from_numpy(d["gt"][:, :3]).float(), torch.from_numpy(d["gt_part"]).long()

def segment_oracle(d, xyz):
    """En yakın GT noktasının parçası. Segmentasyon hatasını İZOLE etmek için —
       'renklendirme fikri iyi mi' ile 'segmenter yeterli mi' ayrı sorular."""
    x = np.asarray(xyz, np.float32)[:, :3]
    c = x.mean(0); x = (x - c) / (np.linalg.norm(x - c, axis=1).max() + 1e-9)
    g = d["gt"][:, :3]; gc = g.mean(0); g = (g - gc) / (np.linalg.norm(g - gc, axis=1).max() + 1e-9)
    _, j = cKDTree(g).query(x, k=1)
    return d["gt_part"][j]

# part-seg kalitesi — tabloyu yorumlamak için ŞART: parça koşullaması bu etikete dayanır,
# etiket kötüyse RePaint-part de düşer (bu gerçekten gözlendi).
def part_color_pointnet(partial, comp, model):   # BASELINE 2 — mevcut yöntemin
    vl = segment(model, partial[:, :3]); cl = segment(model, comp[:, :3])
    P = NUM_PARTS; vr = partial[:, 3:6]
    mean = np.tile(vr.mean(0), (P, 1))           # dayanaksız parça -> global ortalama
    for k in range(P):
        m = vl == k
        if m.any(): mean[k] = vr[m].mean(0)
    return mean[cl]


In [ ]:
# ---------------- D1 · DDPM şeması + RePaint zıplama şeması ----------------
import math

def cosine_betas(T, s=0.008):
    """Nichol & Dhariwal cosine schedule — düşük boyutlu sinyalde linear'dan iyi."""
    t = torch.linspace(0, T, T + 1) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    ab = f / f[0]
    return (1 - ab[1:] / ab[:-1]).clamp(1e-8, 0.999)

class Diffusion:
    """Düz DDPM (eps-tahmini), [-1,1] aralığındaki (N,3) renk alanı üzerinde."""
    def __init__(self, T=200, device=DEV):
        self.T, self.device = T, device
        b = cosine_betas(T).to(device); a = 1.0 - b
        abar = torch.cumprod(a, 0)
        abar_prev = torch.cat([torch.ones(1, device=device), abar[:-1]])
        self.betas, self.alphas, self.abar, self.abar_prev = b, a, abar, abar_prev
        self.sqrt_abar, self.sqrt_1mabar = abar.sqrt(), (1 - abar).sqrt()
        self.post_var = b * (1 - abar_prev) / (1 - abar)            # q(x_{t-1}|x_t,x_0)
        self.post_c0  = b * abar_prev.sqrt() / (1 - abar)
        self.post_ct  = (1 - abar_prev) * a.sqrt() / (1 - abar)

    def q_sample(self, x0, t, noise=None):
        noise = torch.randn_like(x0) if noise is None else noise
        sa = self.sqrt_abar[t].view(-1, *([1] * (x0.dim() - 1)))
        sb = self.sqrt_1mabar[t].view(-1, *([1] * (x0.dim() - 1)))
        return sa * x0 + sb * noise

    def p_sample(self, eps, x_t, t, generator=None):
        x0 = ((x_t - self.sqrt_1mabar[t] * eps) / self.sqrt_abar[t]).clamp(-1, 1)
        mean = self.post_c0[t] * x0 + self.post_ct[t] * x_t
        if t == 0: return mean, x0
        z = torch.randn(x_t.shape, device=x_t.device, dtype=x_t.dtype, generator=generator)
        return mean + self.post_var[t].sqrt() * z, x0

    def forward_jump(self, x, t, generator=None):        # RePaint time-travel: x_t -> x_{t+1}
        z = torch.randn(x.shape, device=x.device, dtype=x.dtype, generator=generator)
        return self.alphas[t].sqrt() * x + self.betas[t].sqrt() * z

def get_schedule_jump(T, jump_length=10, jump_n_sample=5):
    """RePaint'in resampling ('time-travel') şeması — resmî repo ile aynı.

    Dönen listede ardışık AZALAN çift = ters difüzyon adımı, ARTAN çift = ileri zıplama.
    Zıplamalar üretilen bölgenin bilinen bölgeyle ANLAMCA uyumlanmasını sağlar; makalenin
    ana katkısı bu (zıplamasız versiyon sadece dokuca uyar)."""
    jumps = {j: jump_n_sample - 1 for j in range(0, T - jump_length, jump_length)}
    t, ts = T, []
    while t >= 1:
        t -= 1; ts.append(t)
        if jumps.get(t, 0) > 0:
            jumps[t] -= 1
            for _ in range(jump_length):
                t += 1; ts.append(t)
    ts.append(-1)
    return ts

DIF = Diffusion(T=200)
print("T =", DIF.T, "| RePaint adım sayısı (j=10, U=3):", len(get_schedule_jump(DIF.T, 10, 3)))

In [ ]:
# ---------------- D2 · parça-koşullu denoiser ----------------
def knn_graph(xyz, k, chunk=4096):
    """(B,N,3) -> (B,N,k) komşu indisleri (kendisi hariç), sorgu üzerinden parçalı."""
    B, N, _ = xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=xyz.device)
    kk = min(k + 1, N)
    for s in range(0, N, chunk):
        d = torch.cdist(xyz[:, s:s + chunk], xyz)
        idx = d.topk(kk, dim=-1, largest=False).indices[:, :, 1:]
        if idx.shape[-1] < k:
            idx = idx[..., [i % idx.shape[-1] for i in range(k)]]
        out[:, s:s + chunk] = idx
    return out

def _gather_nb(h, idx):                        # h (B,N,C), idx (B,N,k) -> (B,N,k,C)
    B, N, C = h.shape; k = idx.shape[-1]
    off = (torch.arange(B, device=h.device) * N).view(B, 1, 1)
    return h.reshape(B * N, C)[(idx + off).reshape(-1)].reshape(B, N, k, C)

def part_pool(h, part, P):
    """h'nin PARÇA İÇİ ortalaması, her noktaya geri yayılır. İşte 'part-based' burada:
       bir parçanın (enjekte edilmiş) görünür renkleri kendi eksik noktalarını sürer."""
    B, N, C = h.shape
    flat = (part + torch.arange(B, device=h.device).view(B, 1) * P).reshape(-1)
    s = torch.zeros(B * P, C, device=h.device, dtype=h.dtype).index_add_(0, flat, h.reshape(-1, C))
    n = torch.zeros(B * P, 1, device=h.device, dtype=h.dtype).index_add_(
        0, flat, torch.ones(B * N, 1, device=h.device, dtype=h.dtype))
    return torch.gather((s / n.clamp(min=1.0)).reshape(B, P, C), 1, part.unsqueeze(-1).expand(B, N, C))

def timestep_embedding(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    return torch.cat([a.sin(), a.cos()], -1)

class Block(nn.Module):
    """EdgeConv (yerel geometri) + parça havuzu + global havuz, t ile FiLM'lenir."""
    def __init__(self, w, part_cond=True):
        super().__init__(); self.part_cond = part_cond
        self.edge = nn.Sequential(nn.Linear(2 * w + 4, w), nn.GELU(), nn.Linear(w, w))
        ctx = w * (3 if part_cond else 2)
        self.fuse = nn.Sequential(nn.LayerNorm(ctx), nn.Linear(ctx, w), nn.GELU(), nn.Linear(w, w))
        self.film = nn.Linear(w, 2 * w)
    def forward(self, h, idx, rel, part, P, temb):
        hj = _gather_nb(h, idx); hi = h.unsqueeze(2).expand_as(hj)
        e = self.edge(torch.cat([hi, hj - hi, rel], -1)).max(2).values
        g = h.max(1, keepdim=True).values.expand_as(h)
        c = [e, g] + ([part_pool(h, part, P)] if self.part_cond else [])
        d = self.fuse(torch.cat(c, -1))
        sc, sh = self.film(temb).unsqueeze(1).chunk(2, -1)
        return h + d * (1 + sc) + sh

class PartColorDenoiser(nn.Module):
    """Nokta başına renk alanı için eps-tahmini; xyz (+ parça) ile koşullu.

    Permütasyona eşdeğişken ve N'den bağımsız: 2048 noktalı GT bulutlarında eğitilir,
    ~7k noktalı PoinTr birleşim bulutunda çalıştırılır.
    `part_cond=False` -> parça-kör (vanilla RePaint) ablasyonu.
    """
    def __init__(self, num_parts, width=128, k=16, n_blocks=3, part_cond=True):
        super().__init__()
        self.P, self.k, self.part_cond, self.width = num_parts, k, part_cond, width
        cin = 3 + 3 + (num_parts if part_cond else 0)          # xyz, c_t, parça one-hot
        self.inp = nn.Linear(cin, width)
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList([Block(width, part_cond) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 3))
    def build_ctx(self, xyz, part):
        """Geometri her difüzyon adımında SABİT -> kNN grafiği bir kez kurulur (büyük hızlanma)."""
        idx = knn_graph(xyz, self.k)
        rel = _gather_nb(xyz, idx) - xyz.unsqueeze(2)
        scale = rel.norm(dim=-1).mean(dim=(1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
        rel = torch.cat([rel / scale, rel.norm(dim=-1, keepdim=True) / scale], -1)  # yoğunluktan bağımsız
        return dict(xyz=xyz, idx=idx, rel=rel, part=part,
                    onehot=torch.nn.functional.one_hot(part, self.P).float())
    def forward(self, c_t, t, ctx):
        B = c_t.shape[0]
        f = [ctx["xyz"], c_t] + ([ctx["onehot"]] if self.part_cond else [])
        h = self.inp(torch.cat(f, -1))
        temb = self.temb(timestep_embedding(t.expand(B) if t.dim() else t.repeat(B), self.width))
        for blk in self.blocks:
            h = blk(h, ctx["idx"], ctx["rel"], ctx["part"], self.P, temb)
        return self.out(h)


In [ ]:
# ---------------- D4 · RePaint çıkarımı (parça bazlı) ----------------
@torch.no_grad()
def repaint_colors(model, dif, xyz, part, known, known_rgb, *, jump_length=10,
                   jump_n_sample=3, adaptive_jumps=True, seed=0, device=DEV, return_diag=False):
    """Parça bazlı RePaint: doldurulan noktaların rengini inpaint eder.

    Lugmayr et al. (2022)'ye sadık: her ters adımda BİLİNEN renkler doğru gürültü
    seviyesinde geri enjekte edilir, şema yukarı zıplayarak üretilen bölgenin onlarla
    uyumlanmasını sağlar:

        x_{t-1} = m . q(x_bilinen, t-1)  +  (1-m) . p_theta(x_t)

    "Part-based" üç noktada:
      1. denoiser özellikleri PARÇA İÇİNDE havuzlar -> bir parçanın bilinen renkleri kendi
         eksik noktalarını sürer (parça-ortalaması kuralının öğrenilmiş hâli);
      2. dayanaklar parça bazlı sayılır — görünür noktası SIFIR olan parça *anchorless*
         işaretlenir ve komşusundan renk sızdırmak yerine öğrenilmiş önselden doldurulur;
      3. resampling bütçesi en kötü parçanın görünürlüğüne göre uyarlanır.

    xyz (N,3), part (N,), known (N,) bool, known_rgb (N,3) [0,1] -> (N,3) [0,1].
    """
    g = torch.Generator(device=device).manual_seed(seed)
    xyz_t = torch.as_tensor(xyz).float().unsqueeze(0).to(device)
    prt = torch.as_tensor(part).long().unsqueeze(0).to(device)
    m = torch.as_tensor(known).bool().view(1, -1, 1).to(device)
    c0 = (torch.as_tensor(known_rgb).float().unsqueeze(0).to(device) * 2 - 1) * m

    # --- parça bazlı dayanak muhasebesi ---
    pid = prt[0]
    vis = np.array([(m[0, :, 0][pid == p].float().mean().item() if (pid == p).any() else np.nan)
                    for p in range(model.P)])
    present = ~np.isnan(vis)
    anchorless = [p for p in range(model.P) if present[p] and vis[p] == 0.0]
    if adaptive_jumps and present.any():
        worst = float(np.nanmin(np.where(present, vis, np.nan)))
        jump_n_sample = int(np.clip(round(jump_n_sample * (1.5 - worst)), 1, 2 * jump_n_sample))

    model.eval().to(device)
    ctx = model.build_ctx(xyz_t, prt)
    x = torch.randn(1, xyz_t.shape[1], 3, device=device, generator=g)

    ts = get_schedule_jump(dif.T, jump_length, jump_n_sample)
    for t_cur, t_next in zip(ts[:-1], ts[1:]):
        if t_next < t_cur:                                          # --- ters adım
            eps = model(x, torch.tensor(t_cur, device=device), ctx)
            x_unknown, _ = dif.p_sample(eps, x, t_cur, generator=g)
            if t_cur > 0:
                noise = torch.randn(x.shape, device=device, generator=g)
                x_known = dif.q_sample(c0, torch.tensor([t_cur - 1], device=device), noise)
            else:
                x_known = c0
            x = torch.where(m, x_known, x_unknown)                  # maske = parça bazlı maskelerin birleşimi
        else:                                                       # --- ileri zıplama
            x = dif.forward_jump(x, t_cur, generator=g)

    out = ((x[0] + 1) / 2).clamp(0, 1).cpu().numpy()
    kn = np.asarray(known, bool)
    out[kn] = np.asarray(known_rgb, np.float32)[kn]                 # görünür renkler birebir korunur
    if return_diag:
        return out, dict(part_visibility=vis, anchorless_parts=anchorless,
                         jump_n_sample=jump_n_sample, n_steps=len(ts))
    return out


def make_repaint_input(d, comp, seg_model, drop_part=None, oracle_seg=False):
    """Birleşim bulutu = görünür partial (renk BİLİNEN) + PoinTr'ın doldurduğu noktalar (BİLİNMEYEN).

    oracle_seg=True -> parça etiketleri PointNet yerine en yakın GT'den (segmentasyon hatasız tavan).
    drop_part verilirse o parçanın bütün görünür noktaları da maskelenir -> *anchorless* senaryo.
    Düşürme HER ZAMAN GT etiketine göre yapılır: aksi hâlde segmenter o parçayı hiç tahmin
    etmediğinde 'hiçbir şey düşmez' ve deney sessizce anlamsızlaşır (bu tuzağa bir kez düşüldü).
    """
    partial = d["partial"]
    xyz = np.concatenate([partial[:, :3], comp], 0).astype(np.float32)
    c = xyz.mean(0); xyz = ((xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-9)).astype(np.float32)
    known = np.zeros(len(xyz), bool); known[:len(partial)] = True
    rgb = np.zeros((len(xyz), 3), np.float32); rgb[:len(partial)] = partial[:, 3:6]
    part = segment_oracle(d, xyz) if oracle_seg else segment(seg_model, xyz)
    if drop_part is not None:
        known &= (segment_oracle(d, xyz) != drop_part); rgb[~known] = 0
    return dict(xyz=xyz, rgb=rgb, known=known, part=part, n_vis=len(partial))

print("RePaint çıkarımı hazır")

In [ ]:
# eğitim döngüsü (D3'ün fonksiyon kısmı — burada modelleri biz kuruyoruz)
def train_color_ddpm(model, dif, clouds, epochs=300, bs=8, lr=2e-4, device=DEV, log=100):
    """Koşulsuz DDPM eğitimi — occlusion maskesi HİÇ görülmez."""
    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    n = len(clouds)
    for ep in range(epochs):
        perm = np.random.permutation(n); tot = 0.0
        for s in range(0, n, bs):
            b = [clouds[i] for i in perm[s:s + bs]]
            xyz = torch.stack([torch.as_tensor(d["xyz"]) for d in b]).float().to(device)
            rgb = torch.stack([torch.as_tensor(d["rgb"]) for d in b]).float().to(device)
            prt = torch.stack([torch.as_tensor(d["part"]) for d in b]).long().to(device)
            x0 = rgb * 2 - 1
            t = torch.randint(0, dif.T, (len(b),), device=device)
            noise = torch.randn_like(x0)
            loss = ((model(dif.q_sample(x0, t, noise), t, model.build_ctx(xyz, prt)) - noise) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        if log and (ep % log == 0 or ep == epochs - 1):
            print(f"    ep{ep:4d} eps-MSE {tot/n:.4f}", flush=True)
    return tot / n
print("eğitim döngüsü hazır")

---
# E · Benchmark — bu kategori, üç zorluk

Sıra: **part-seg eğit → iki DDPM'i BİR KEZ eğit → üç zorlukta değerlendir.**
DDPM occlusion maskesi görmediği için zorluk başına yeniden eğitim gerekmiyor —
RePaint'in asıl satış argümanı bu ve tablo onu doğrudan gösteriyor.

**Kopmaya dayanıklı.** Eğitilmiş modeller Drive'a yazılır; her test modelinin sonucu
hesaplandığı anda Drive'daki `results.jsonl`'a eklenir. Runtime düşerse aynı hücreyi
tekrar çalıştır — eğitim atlanır, hesaplanmış modeller atlanır, kaldığı yerden sürer.

In [ ]:
# ---------------- Drive checkpoint + sonuç önbelleği ----------------
import json, time
try:
    from google.colab import drive; drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/pcc_bench"
except Exception as e:
    CKPT_DIR = "/content/pcc_bench"
    print(f"⚠️ Drive bağlanamadı ({type(e).__name__}) -> {CKPT_DIR} (kopunca kaybolur)")
os.makedirs(CKPT_DIR, exist_ok=True)
RESULTS = os.path.join(CKPT_DIR, "results.jsonl")
print("checkpoint:", CKPT_DIR)

def load_results():
    if not os.path.exists(RESULTS): return []
    out = []
    for ln in open(RESULTS):
        ln = ln.strip()
        if ln:
            try: out.append(json.loads(ln))
            except json.JSONDecodeError: pass
    return out

def append_result(rec):
    with open(RESULTS, "a") as f:
        f.write(json.dumps(rec) + "\n")

def ckpt_path(name): return os.path.join(CKPT_DIR, f"{RUN_CATEGORY}_{name}.pt")

def save_model(name, model, meta):
    torch.save({"sd": model.state_dict(), "meta": meta}, ckpt_path(name))
    print(f"  💾 {name}")

def load_model(name, model, meta):
    p = ckpt_path(name)
    if FORCE_RETRAIN or not os.path.exists(p): return False
    try: z = torch.load(p, map_location=DEV, weights_only=False)
    except Exception: return False
    if z.get("meta") != meta:
        print(f"  ⛔ {name} checkpoint config'i farklı -> yeniden eğitilecek"); return False
    model.load_state_dict(z["sd"]); model.to(DEV); return True

_done = {(r["category"], r["difficulty"], r["model"]) for r in load_results()}
print(f"önbellekte {len(_done)} model-sonucu var")

In [ ]:
# ---------------- ANA DÖNGÜ: 3 kategori × 3 zorluk ----------------
from scipy.spatial import cKDTree
DDPM_EPOCHS, SEG_EPOCHS, DDPM_ARCH = 300, 60, dict(width=128, k=16, n_blocks=3)
KEYS = ["NN-copy", "part-mean", "RePaint-vanilla", "RePaint-part",
        "RePaint-part (oracle seg)", "oracle ceiling"]

def cloud3d(xyz, rgb):
    return wandb.Object3D(np.concatenate([xyz, np.clip(rgb, 0, 1) * 255], 1).astype(np.float32))

T_START = time.time()
for category in RUN_CATEGORIES:
    print(f"\n{'='*62}\n  {category.upper()}   (geçen {(time.time()-T_START)/60:.0f} dk)\n{'='*62}",
          flush=True)
    DATA = load_split(category, DIFFICULTIES[0])
    NUM_PARTS = len(PART_NAMES[category])
    n_test = max(1, int(len(DATA) * TEST_FRAC))
    TRAIN_IDX = list(range(len(DATA) - n_test))
    TEST_IDX  = list(range(len(DATA) - n_test, len(DATA)))
    print(f"  {len(DATA)} model | train {len(TRAIN_IDX)} / test {len(TEST_IDX)} "
          f"| {NUM_PARTS} parça {PART_NAMES[category]}", flush=True)

    # --- frame: her kategoride yeniden doğrula ---
    AXIS_PERM, AXIS_SIGN, best, ref = find_frame(DATA, TRAIN_IDX)
    verdict = "OK" if best < 0.6 * ref else ("ZAYIF" if best < ref else "BOZUK")
    print(f"  frame perm={list(AXIS_PERM)} sign={list(AXIS_SIGN)} | chamfer {best:.4f} "
          f"vs ham partial {ref:.4f} -> {verdict}", flush=True)
    if verdict == "BOZUK":
        print("  !! PoinTr bu kategoride şekli tanımıyor — sonuçlar güvenilmez", flush=True)

    META = dict(cat=category, n_train=len(TRAIN_IDX), n_pts=N_PTS, parts=NUM_PARTS,
                ddpm_ep=DDPM_EPOCHS, seg_ep=SEG_EPOCHS, T=DIF.T, **DDPM_ARCH)

    seg_model = PointNetPartSeg(NUM_PARTS).to(DEV)
    if load_model(category, "seg", seg_model, META):
        print("  part-seg: checkpoint'ten", flush=True)
    else:
        print("  ── part-seg eğitimi", flush=True)
        train_partseg(seg_model, DataLoader(_GTPartDS([DATA[i] for i in TRAIN_IDX]),
                                            batch_size=16, shuffle=True), epochs=SEG_EPOCHS)
        save_model(category, "seg", seg_model, META)
    SEG_ACC = float(np.mean([(segment(seg_model, DATA[i]["gt"][:, :3]) == DATA[i]["gt_part"]).mean()
                             for i in TEST_IDX]))
    print(f"  part-seg TEST doğruluğu: {SEG_ACC*100:.1f}%", flush=True)

    TC = [dict(xyz=DATA[i]["gt"][:, :3], rgb=DATA[i]["gt"][:, 3:6], part=DATA[i]["gt_part"])
          for i in TRAIN_IDX]
    ddpm_part = PartColorDenoiser(NUM_PARTS, part_cond=True, **DDPM_ARCH)
    if load_model(category, "ddpm_part", ddpm_part, META):
        print("  DDPM part-conditioned: checkpoint'ten", flush=True)
    else:
        print("  ── DDPM (part-conditioned)", flush=True)
        train_color_ddpm(ddpm_part, DIF, TC, epochs=DDPM_EPOCHS)
        save_model(category, "ddpm_part", ddpm_part, META)

    ddpm_van = PartColorDenoiser(NUM_PARTS, part_cond=False, **DDPM_ARCH)
    if load_model(category, "ddpm_van", ddpm_van, META):
        print("  DDPM part-blind: checkpoint'ten", flush=True)
    else:
        print("  ── DDPM (part-blind)", flush=True)
        train_color_ddpm(ddpm_van, DIF, TC, epochs=DDPM_EPOCHS)
        save_model(category, "ddpm_van", ddpm_van, META)

    # ---------------- üç zorlukta değerlendir ----------------
    for difficulty in DIFFICULTIES:
        DATA = load_split(category, difficulty)
        idxs = TEST_IDX if EVAL_N is None else TEST_IDX[:EVAL_N]
        todo = [i for i in idxs if (category, difficulty, DATA[i]["model_id"]) not in _done]
        print(f"\n  ── {difficulty}: {len(idxs)} test modeli, {len(todo)} yapılacak", flush=True)

        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                         id=f"{category}-{difficulty}", name=f"{category}-{difficulty}",
                         group=category, job_type="eval", resume="allow", reinit=True,
                         config=dict(category=category, synset=CATEGORIES[category],
                                     difficulty=difficulty,
                                     crop_ratio={"simple": .25, "moderate": .5, "hard": .75}[difficulty],
                                     n_models=len(DATA), n_train=len(TRAIN_IDX), n_test=len(idxs),
                                     n_pts=N_PTS, n_seeds=N_SEEDS, ddpm_epochs=DDPM_EPOCHS,
                                     seg_epochs=SEG_EPOCHS, T=DIF.T, axis_perm=list(AXIS_PERM),
                                     axis_sign=list(AXIS_SIGN), pointr_in=POINTR_IN,
                                     dataset=HF_DATA, partseg_acc=SEG_ACC, **DDPM_ARCH))
        run.log({"partseg/test_acc": SEG_ACC})

        t0 = time.time()
        for n, i in enumerate(todo):
            d = DATA[i]; gt, partial = d["gt"], d["partial"]
            comp = complete_of(d)
            ch_in  = chamfer_l1(partial[:, :3], gt[:, :3])
            ch_out = chamfer_l1(comp, gt[:, :3])
            _, gi = cKDTree(gt[:, :3]).query(comp, k=1)
            true_rgb = gt[gi, 3:6]; m = d["miss"][gi]
            if not m.any():
                continue
            rec = {"category": category, "difficulty": difficulty, "model": d["model_id"],
                   "chamfer_in": ch_in, "chamfer_out": ch_out}
            rec["NN-copy"]   = float(deltaE(nn_color(partial, comp)[m], true_rgb[m]).mean())
            rec["part-mean"] = float(deltaE(part_color_pointnet(partial, comp, seg_model)[m],
                                            true_rgb[m]).mean())
            pf = np.stack([gt[d["gt_part"] == p, 3:6].mean(0) if (d["gt_part"] == p).any()
                           else gt[:, 3:6].mean(0) for p in range(NUM_PARTS)])
            rec["oracle ceiling"] = float(deltaE(pf[d["gt_part"][gi]][m], true_rgb[m]).mean())
            inp    = make_repaint_input(d, comp, seg_model)
            inp_or = make_repaint_input(d, comp, seg_model, oracle_seg=True)
            for tag, mdl, ip in [("RePaint-vanilla", ddpm_van, inp),
                                 ("RePaint-part", ddpm_part, inp),
                                 ("RePaint-part (oracle seg)", ddpm_part, inp_or)]:
                sv = [deltaE(repaint_colors(mdl, DIF, ip["xyz"], ip["part"], ip["known"], ip["rgb"],
                                            seed=1000 * n + k)[ip["n_vis"]:][m], true_rgb[m]).mean()
                      for k in range(N_SEEDS)]
                rec[tag] = float(np.mean(sv))
            append_result(rec); _done.add((category, difficulty, d["model_id"]))
            run.log({f"model/{k}": v for k, v in rec.items() if isinstance(v, float)})
            el = time.time() - t0
            print(f"    [{n+1}/{len(todo)}] {d['model_id'][:10]} NN {rec['NN-copy']:5.2f} | "
                  f"pm {rec['part-mean']:5.2f} | RP {rec['RePaint-part']:5.2f}  "
                  f"({el/(n+1):.0f}s/model)", flush=True)

        rs = [r for r in load_results()
              if r["category"] == category and r["difficulty"] == difficulty]
        if rs:
            agg = {"chamfer/partial_to_gt": float(np.mean([r["chamfer_in"] for r in rs])),
                   "chamfer/completion_to_gt": float(np.mean([r["chamfer_out"] for r in rs]))}
            agg["chamfer/improvement_x"] = (agg["chamfer/partial_to_gt"] /
                                            max(agg["chamfer/completion_to_gt"], 1e-9))
            for k in KEYS:
                agg[f"dE/{k}"] = float(np.mean([r[k] for r in rs]))
            agg["dE/repaint_part_vs_nn_x"] = agg["dE/NN-copy"] / max(agg["dE/RePaint-part"], 1e-9)
            agg["n_evaluated"] = len(rs)
            run.log(agg); run.summary.update(agg)
            run.log({"per_model": wandb.Table(
                columns=["model", "chamfer_in", "chamfer_out"] + KEYS,
                data=[[r["model"], r["chamfer_in"], r["chamfer_out"]] + [r[k] for k in KEYS]
                      for r in rs])})
            print(f"    => NN {agg['dE/NN-copy']:.2f} | part-mean {agg['dE/part-mean']:.2f} | "
                  f"RePaint-part {agg['dE/RePaint-part']:.2f} | tavan {agg['dE/oracle ceiling']:.2f}"
                  f" | chamfer {agg['chamfer/improvement_x']:.2f}x (n={len(rs)})", flush=True)

        d0 = DATA[idxs[0]]; comp0 = complete_of(d0)
        inp0 = make_repaint_input(d0, comp0, seg_model)
        rp0 = repaint_colors(ddpm_part, DIF, inp0["xyz"], inp0["part"], inp0["known"],
                             inp0["rgb"], seed=0)
        run.log({"cloud/gt": cloud3d(d0["gt"][:, :3], d0["gt"][:, 3:6]),
                 "cloud/partial": cloud3d(d0["partial"][:, :3], d0["partial"][:, 3:6]),
                 "cloud/nn_copy": cloud3d(comp0, nn_color(d0["partial"], comp0)),
                 "cloud/repaint_part": cloud3d(comp0, rp0[inp0["n_vis"]:])})
        run.finish()

print(f"\nBENCHMARK BİTTİ — toplam {(time.time()-T_START)/60:.0f} dk")

---
# F · Rapor

Bütün kategoriler bittikten sonra çalıştır. Drive'daki `results.jsonl`'dan
ızgara tablosunu kurar, W&B'ye özet run olarak yazar ve indirilebilir bir HTML rapor üretir.

In [ ]:
# ---------------- özet + rapor ----------------
import pandas as pd
rs = load_results()
if not rs:
    print("henüz sonuç yok")
else:
    df = pd.DataFrame(rs)
    grid = df.groupby(["category","difficulty"]).agg(
        n=("model","count"), chamfer_in=("chamfer_in","mean"), chamfer_out=("chamfer_out","mean"),
        **{k: (k,"mean") for k in KEYS}).reset_index()
    grid["chamfer_x"] = grid["chamfer_in"] / grid["chamfer_out"]
    order = {"simple":0,"moderate":1,"hard":2}
    grid = grid.sort_values(["category", "difficulty"], key=lambda s: s.map(order).fillna(s))
    pd.set_option("display.width", 200, "display.max_columns", 50)
    print(grid.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    summ = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="SUMMARY",
                      id="SUMMARY", resume="allow", job_type="summary", reinit=True)
    summ.log({"grid": wandb.Table(dataframe=grid)})
    for metric in ["NN-copy","part-mean","RePaint-part","oracle ceiling"]:
        summ.log({f"bar/{metric}": wandb.plot.bar(wandb.Table(
            data=[[f"{r.category}-{r.difficulty}", float(getattr(r, metric.replace(' ','_').replace('-','_'), r[metric] if isinstance(r, dict) else 0))]
                  for _, r in grid.iterrows()] if False else
                 [[f"{r['category']}-{r['difficulty']}", float(r[metric])] for _, r in grid.iterrows()],
            columns=["config", metric]), "config", metric, title=f"dE {metric}")})
    summ.finish()

    grid.to_csv("/content/benchmark_grid.csv", index=False)
    html = grid.to_html(index=False, float_format=lambda x: f"{x:.3f}")
    open("/content/benchmark_report.html","w").write(
        "<meta charset='utf-8'><style>body{font-family:system-ui;margin:40px;max-width:1100px}"
        "table{border-collapse:collapse;width:100%;font-size:13px}th,td{border-bottom:1px solid #ddd;"
        "padding:7px 9px;text-align:right}th{text-align:left;background:#f6f8fa}td:first-child,"
        "th:first-child{text-align:left}</style>"
        f"<h1>Colored point-cloud completion — benchmark</h1>"
        f"<p>Dataset: <code>{HF_DATA}</code> · ΔE(Lab), eksik bölge · düşük = iyi</p>{html}")
    print("\n/content/benchmark_grid.csv ve /content/benchmark_report.html yazıldı")

---
## W&B'de neye bakacaksın

**Runs** ekranında 9 run (`airplane-simple` … `chair-hard`) + `SUMMARY`. `group` sütunu
kategoriye göre katlar.

Asıl sorular:

1. `dE/part-mean` tavana (`dE/oracle ceiling`) yapıştı mı? Yapıştıysa deterministik kural
   sınırına dayanmış demektir.
2. `dE/RePaint-part` onun **altına** indi mi? Katkı buradan okunur.
3. `dE/RePaint-part (oracle seg)` ile farkı = **part-seg hatasının bedeli**. Büyükse
   bir sonraki iş difüzyon değil segmenter.
4. `chamfer/improvement_x` zorluk arttıkça nasıl düşüyor — PoinTr'ın kendi limiti.
5. `dE/repaint_part_vs_nn_x` — "hazır yöntemle yapılana" karşı kaç kat.

**Object3D** panellerinde bulutları döndürüp NN-kopya ile RePaint-part'ı görsel
kıyaslayabilirsin; tabloda görünmeyen renk sızması orada görünür.

Rapor hücresi ayrıca `benchmark_grid.csv` üretiyor — makalenin tablosu doğrudan oradan gelir.